# Milestone 2 — Data Quality dan Profiling Indonesia News

**Nama:** Naufal Zainul Zaki  
**NIM:** 202310370311217  
**GitHub:** Falz4i

Disesuaikan dari [template dosen](https://github.com/UMM-GURU/tugas1-abd-2026/blob/main/notebooks/02_data_cleaning.ipynb), diperiksa 26 Oktober 2026. Data mentah tetap menjadi sumber utama; notebook ini dapat dijalankan tanpa variabel dari notebook sebelumnya.

Urutan: profil awal DuckDB → audit missing → normalisasi dan filter domain → deteksi IQR → deduplikasi → validasi → Parquet. Polars menggunakan lazy scanning; hanya tabel ringkasan yang dikumpulkan ke memori.


In [1]:
from pathlib import Path
import hashlib
import json
import duckdb
import polars as pl
from IPython.display import display, Markdown

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').is_dir())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'archive'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = PROCESSED_DIR / 'indonesia_news_clean.parquet'
RAW_STAGE = PROCESSED_DIR / '_news_raw_profile.parquet'
NORMALIZED_STAGE = PROCESSED_DIR / '_news_normalized.parquet'
CANDIDATE_PATH = PROCESSED_DIR / '_news_clean_candidate.parquet'
con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute('SET threads=2')
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(70)
print('Polars:', pl.__version__, 'DuckDB:', duckdb.__version__)
FILE_CONFIG = [
    {'name': 'cnn.csv', 'separator': '|', 'source': 'CNN Indonesia'},
    {'name': 'tempo.csv', 'separator': '|', 'source': 'Tempo'},
    {'name': 'tempo010619310520.csv', 'separator': ';', 'source': 'Tempo'},
]
for item in FILE_CONFIG:
    if not (RAW_DIR / item['name']).is_file():
        raise FileNotFoundError(RAW_DIR / item['name'])
EXPECTED = ['content', 'datetime', 'tags', 'title']

def scan_file(item):
    return (pl.scan_csv(
        RAW_DIR / item['name'], separator=item['separator'],
        schema_overrides={col: pl.String for col in EXPECTED},
        null_values=['', 'null', 'None', 'NA', 'N/A'],
        row_index_name='source_row', low_memory=True,
        ignore_errors=False, truncate_ragged_lines=False, encoding='utf8'
    ).with_columns(
        pl.lit(item['source']).alias('source'),
        pl.lit(item['name']).alias('source_file'),
        pl.col('datetime').str.strip_chars().str.replace_all(',', '.')
          .cast(pl.Float64, strict=False).round(0).cast(pl.Int64, strict=False).alias('datetime_ms')
    ).with_columns(
        pl.when(pl.col('datetime_ms').is_between(1388534400000, 1609459199999))
          .then(pl.col('datetime_ms')).otherwise(None).alias('valid_datetime_ms')
    ).with_columns(pl.from_epoch('valid_datetime_ms', time_unit='ms').alias('published_at')))

# CSV parsing is strict: malformed rows stop execution instead of being silently truncated.
pl.concat([scan_file(item) for item in FILE_CONFIG]).sink_parquet(RAW_STAGE, compression='zstd')
raw_scan = pl.scan_parquet(RAW_STAGE)
con.from_parquet(str(RAW_STAGE)).create_view('raw_news', replace=True)
raw_count = con.sql('SELECT count(*) FROM raw_news').fetchone()[0]
assert raw_count > 0
print(f'Baris mentah: {raw_count:,}')
display(raw_scan.head(3).select('source', 'source_file', 'datetime', 'published_at', 'title').collect())


Polars: 1.44.2 DuckDB: 1.5.5


Baris mentah: 988,814


source,source_file,datetime,published_at,title
str,str,str,datetime[μs],str
"""CNN Indonesia""","""cnn.csv""","""1407141300000""",2014-08-04 08:35:00,""" Pengintai AS Terpaksa Masuk Wilayah Swedia …"
"""CNN Indonesia""","""cnn.csv""","""1407148440000""",2014-08-04 10:34:00,""" Mutan Kura-kura Remaja Kembali Beraksi …"
"""CNN Indonesia""","""cnn.csv""","""1407151380000""",2014-08-04 11:23:00,""" Pertemuan Dua Raksasa Inggris """


## 1. Profiling awal DuckDB SUMMARIZE

Profil sebelum cleaning meliputi empat kolom sumber dan tanggal hasil parsing. Null pada `published_at` berarti waktu tidak dapat dibaca atau berada di luar domain 2014–2020. Timestamp tanpa zona diperlakukan sebagai waktu epoch UTC, tanpa mengasumsikan jam lokal redaksi.

In [2]:
raw_profile = con.sql('SUMMARIZE SELECT content, datetime, tags, title, published_at FROM raw_news').pl()
display(raw_profile)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,i32,str,str,str,i64,"decimal[9,2]"
"""content""","""VARCHAR""",""" , CNN Indonesia -- Gelaran sidang lanjutan P…","""”Wanhong ini produksinya akan diekspor ke Cina. Karena memang perusaha…",879099,null,null,null,null,null,988814,1.30
"""datetime""","""VARCHAR""","""1,55932E+12""","""datetime""",769157,null,null,null,null,null,988814,0.00
"""tags""","""VARCHAR""",""" ahmed zaki iskandar zulkarnain,lokalisasi,penggusuran""","""zurich,asuransi,digital""",647457,null,null,null,null,null,988814,2.32
"""title""","""VARCHAR""",""" ""","""title""",859169,null,null,null,null,null,988814,0.00
"""published_at""","""TIMESTAMP""","""2014-08-04 08:35:00""","""2020-05-31 16:52:00""",886710,"""2018-02-11 23:47:56.152972""",null,"""2016-09-21 07:19:23.217168""","""2018-05-06 04:57:44.090292""","""2019-08-23 23:48:52.62749""",988814,0.00


## 2. Deteksi missing values

Null dan string berisi whitespace dilaporkan terpisah, lalu dijumlahkan beserta persentasenya. Panjang teks diturunkan dari teks yang ada; nilai judul/isi yang hilang tidak diisi dengan teks buatan atau median.

In [3]:
missing_rows = []
for col in EXPECTED + ['published_at']:
    counts = raw_scan.select(
        pl.col(col).null_count().alias('null_count'),
        pl.col(col).cast(pl.String).str.strip_chars().eq('').fill_null(False).sum().alias('empty_count')
    ).collect(engine='streaming').row(0, named=True)
    missing_rows.append({'column': col, **counts,
                         'missing_total': counts['null_count'] + counts['empty_count'],
                         'missing_percent': 100 * (counts['null_count'] + counts['empty_count']) / raw_count})
missing_summary = pl.DataFrame(missing_rows)
display(missing_summary)


column,null_count,empty_count,missing_total,missing_percent
str,i64,i64,i64,f64
"""content""",12870,1269,14139,1.429895
"""datetime""",0,0,0,0.0
"""tags""",22978,0,22978,2.323794
"""title""",0,176,176,0.017799
"""published_at""",4,0,4,0.000405


## 3. Penanganan missing dan aturan domain

- Judul, isi, dan tanggal wajib tersedia. Baris tidak layak dikeluarkan dari tabel analisis; data mentah tetap tersedia.
- Tags kosong diisi `Tidak diketahui` dan ditandai `tags_missing` agar missing tetap dapat diaudit.
- Whitespace dinormalisasi tanpa mengubah huruf besar/kecil isi dan judul.
- Ambang isi 100 karakter adalah **asumsi operasional**, bukan bukti bahwa semua teks pendek salah. Contoh teks pendek dan jumlah yang terdampak ditampilkan untuk evaluasi.
- Tahun di luar 2014–2020 ditolak sesuai cakupan dataset. Audit tahap dibuat berurutan agar jumlah baris dibuang tidak dihitung dua kali.

In [4]:
def normalize_text(col):
    return pl.col(col).str.replace_all(r'\s+', ' ').str.strip_chars()

normalized = raw_scan.with_columns([normalize_text(c).alias(c) for c in ['title', 'content', 'tags']])
normalized = normalized.with_columns(
    (pl.col('tags').is_null() | pl.col('tags').eq('')).alias('tags_missing'),
    pl.col('title').str.len_chars().alias('title_length'),
    pl.col('content').str.len_chars().alias('content_length'),
).with_columns(
    pl.when(pl.col('tags_missing')).then(pl.lit('Tidak diketahui')).otherwise(pl.col('tags')).alias('tags'),
    (pl.col('title').is_not_null() & pl.col('title').ne('') &
     pl.col('content').is_not_null() & pl.col('content').ne('')).fill_null(False).alias('has_required_text')
)
normalized.sink_parquet(NORMALIZED_STAGE, compression='zstd')
normalized = pl.scan_parquet(NORMALIZED_STAGE)
MIN_CONTENT_CHARS = 100
eligible_text = normalized.filter(pl.col('has_required_text'))
eligible_date = eligible_text.filter(pl.col('published_at').is_not_null())
eligible = eligible_date.filter(pl.col('content_length') >= MIN_CONTENT_CHARS)
stage_counts = normalized.select(
    pl.len().alias('raw'),
    pl.col('has_required_text').sum().alias('required_text'),
    (pl.col('has_required_text') & pl.col('published_at').is_not_null()).sum().alias('valid_date'),
    (pl.col('has_required_text') & pl.col('published_at').is_not_null() &
     (pl.col('content_length') >= MIN_CONTENT_CHARS)).sum().alias('minimum_length'),
).collect(engine='streaming').row(0, named=True)
display(eligible_date.filter(pl.col('content_length') < MIN_CONTENT_CHARS)
        .select('source', 'title', 'content', 'content_length').head(5).collect())
print('Jumlah baris per tahap:', stage_counts)


source,title,content,content_length
str,str,str,u32
"""CNN Indonesia""","""Komisi II Terima Perwakilan Prabowo-Hatta""","""Jakarta, CNN Indonesia --""",25
"""CNN Indonesia""","""ISIS Tidak Akan Berkembang di Indonesia""","""Jakarta, CNN Indonesia --""",25
"""CNN Indonesia""","""BNPKT: ISIS Sudah Menyebar""","""i Indonesia sekarang telah menyebar.""",36
"""CNN Indonesia""","""Khieu Samphan Dihukum Seumur Hidup""","""Phom Phen, Kamboja, CNN Indonesia --""",36
"""CNN Indonesia""","""Israel: Hamas Kembali Tembakkan Roket""","""Gaza, CNN Indonesia --""",22


Jumlah baris per tahap: {'raw': 988814, 'required_text': 974501, 'valid_date': 974497, 'minimum_length': 967113}


## 4. Deteksi dan penanganan outlier dengan IQR

IQR dihitung untuk panjang isi setelah normalisasi dan filter domain, **per media** karena distribusinya berbeda. Pencilan berada di luar Q1 − 1,5×IQR atau Q3 + 1,5×IQR. Pencilan panjang ditandai, bukan otomatis dihapus atau di-winsorize: panjang isi merupakan ukuran faktual dan teks panjang dapat merupakan artikel sah. Tidak dilakukan clipping panjang agar konsisten dengan teks sebenarnya. Contoh pencilan dan jumlahnya membantu pemeriksaan manual.

In [5]:
iqr_bounds = eligible.group_by('source').agg(
    pl.col('content_length').quantile(0.25, interpolation='linear').alias('q1'),
    pl.col('content_length').quantile(0.75, interpolation='linear').alias('q3'),
).with_columns((pl.col('q3') - pl.col('q1')).alias('iqr')).with_columns(
    (pl.col('q1') - 1.5 * pl.col('iqr')).alias('iqr_lower'),
    (pl.col('q3') + 1.5 * pl.col('iqr')).alias('iqr_upper'),
).collect(engine='streaming')
flagged = eligible.join(iqr_bounds.lazy(), on='source', how='left', maintain_order='left').with_columns(
    ((pl.col('content_length') < pl.col('iqr_lower')) |
     (pl.col('content_length') > pl.col('iqr_upper'))).alias('content_length_outlier')
)
display(iqr_bounds)
outlier_audit = flagged.group_by('source').agg(
    pl.len().alias('eligible_rows'), pl.col('content_length_outlier').sum().alias('outlier_rows')
).with_columns((100 * pl.col('outlier_rows') / pl.col('eligible_rows')).alias('outlier_percent')).collect(engine='streaming')
display(outlier_audit)
display(flagged.filter(pl.col('content_length_outlier')).select(
    'source', 'title', 'content_length', 'iqr_lower', 'iqr_upper'
).head(5).collect())


source,q1,q3,iqr,iqr_lower,iqr_upper
str,f64,f64,f64,f64,f64
"""Tempo""",1432.0,2244.0,812.0,214.0,3462.0
"""CNN Indonesia""",1677.0,2583.0,906.0,318.0,3942.0


source,eligible_rows,outlier_rows,outlier_percent
str,u32,u32,f64
"""Tempo""",633832,30560,4.821467
"""CNN Indonesia""",333281,17447,5.234922


source,title,content_length,iqr_lower,iqr_upper
str,str,u32,f64,f64
"""CNN Indonesia""","""Sepakbola versus Komersialisme""",5026,318.0,3942.0
"""CNN Indonesia""","""Line Get's Rich, Jadi Kaya di Dunia Maya.""",3976,318.0,3942.0
"""CNN Indonesia""","""Dilema 'Bola Salju' Ekonomi ASIAN Games""",4644,318.0,3942.0
"""CNN Indonesia""","""Sang Bintang, Terapi Depresi dan Penyekapan Itu""",4703,318.0,3942.0
"""CNN Indonesia""","""Mereka Yang Tumbang Karena Ebola""",6962,318.0,3942.0


## 5. Deduplikasi dan audit overlap Tempo

Dataset tidak menyediakan ID atau URL artikel. Kunci operasional baru adalah **sumber + judul ternormalisasi + SHA-256 isi ternormalisasi**. SHA-256 dipakai untuk membatasi memori; probabilitas tabrakan sangat kecil, tetapi kunci ini tetap bukan ID semantik artikel. Salinan teks persis dengan timestamp berbeda dihitung satu kali, sedangkan revisi isi tetap dipertahankan.

Urutan sumber menentukan salinan yang disimpan: `tempo.csv` didahulukan daripada `tempo010619310520.csv` karena timestamp file tambahan dibulatkan dalam notasi ilmiah. Urutan baris dalam file dipertahankan. Artikel serupa dengan isi berbeda masih mungkin tersisa; tidak dilakukan fuzzy deduplication yang dapat menghapus berita sah.

In [6]:
DEDUP_KEY = ['source', 'title', 'content_sha256']
hashed_path = PROCESSED_DIR / '_news_hashed.parquet'
flagged.with_columns(
    pl.col('content').map_elements(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest(),
                                   return_dtype=pl.String).alias('content_sha256')
).sink_parquet(hashed_path, compression='zstd', maintain_order=True)
hashed = pl.scan_parquet(hashed_path)
con.from_parquet(str(hashed_path)).create_view('eligible_news', replace=True)
overlap_audit = con.sql("""
WITH groups AS (
 SELECT source, title, content_sha256, COUNT(*) AS copies,
        COUNT(DISTINCT source_file) AS file_count,
        COUNT(DISTINCT published_at) AS date_count
 FROM eligible_news GROUP BY source, title, content_sha256
)
SELECT source, SUM(copies - 1) AS duplicate_rows,
       COUNT(*) FILTER (WHERE file_count > 1) AS cross_file_groups,
       COUNT(*) FILTER (WHERE copies > 1 AND date_count > 1) AS groups_with_different_times
FROM groups GROUP BY source ORDER BY source
""").pl()
display(overlap_audit)
display(con.sql("""
SELECT source, title, COUNT(*) AS copies, COUNT(DISTINCT source_file) AS files,
       MIN(published_at) AS earliest, MAX(published_at) AS latest
FROM eligible_news GROUP BY source, title, content_sha256
HAVING COUNT(DISTINCT source_file) > 1
ORDER BY copies DESC, title LIMIT 5
""").pl())
hashed.unique(subset=DEDUP_KEY, keep='first', maintain_order=True).select(
    'source', 'source_file', 'source_row', 'published_at', 'title', 'tags', 'content',
    'title_length', 'content_length', 'tags_missing', 'content_sha256',
    'content_length_outlier', 'iqr_lower', 'iqr_upper'
).sink_parquet(CANDIDATE_PATH, compression='zstd', maintain_order=True)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

source,duplicate_rows,cross_file_groups,groups_with_different_times
str,"decimal[38,0]",i64,i64
"""CNN Indonesia""",837,0,36
"""Tempo""",118248,117022,116506


source,title,copies,files,earliest,latest
str,str,i64,i64,datetime[μs],datetime[μs]
"""Tempo""","""10 Persen Hotel-Resto di Yogyakarta Beroperasi Lagi Juni""",3,2,2020-05-31 13:00:00,2020-05-31 14:02:00
"""Tempo""","""100 Juta Unduhan, Aplikasi Edit Video Android Ini Bawa Malware""",3,2,2020-05-30 06:26:40,2020-05-30 06:53:00
"""Tempo""","""102 Daerah Dibuka, Doni Monardo: Penentunya Bupati/Wali Kota""",3,2,2020-05-30 13:27:00,2020-05-30 14:46:40
"""Tempo""","""102 Daerah Penuhi 11 Indikator Aman Covid-19""",3,2,2020-05-30 09:13:20,2020-05-30 10:10:00
"""Tempo""","""12 Pelanggar PSBB di Jalan Raya Bogor karena Tak Pakai Masker""",3,2,2020-05-28 06:18:00,2020-05-28 07:13:20


## 6. Validasi data dasar dan profil sesudah cleaning

Validasi memeriksa teks kosong, tanggal, sumber, konsistensi panjang teks, keunikan kunci, dan konsistensi flag IQR. Jumlah pencilan boleh lebih dari nol karena kebijakannya mempertahankan dan menandai, sedangkan **pelanggaran aturan** wajib nol. Berkas hasil sebelumnya baru diganti setelah semua assertion lolos.

In [7]:
con.from_parquet(str(CANDIDATE_PATH)).create_view('clean_candidate', replace=True)
validation = con.sql("""
SELECT COUNT(*) AS row_count,
 COUNT(*) FILTER (WHERE title IS NULL OR TRIM(title) = '') AS invalid_title,
 COUNT(*) FILTER (WHERE content IS NULL OR length(content) < 100) AS invalid_content,
 COUNT(*) FILTER (WHERE published_at IS NULL OR published_at < TIMESTAMP '2014-01-01'
                  OR published_at >= TIMESTAMP '2021-01-01') AS invalid_date,
 COUNT(*) FILTER (WHERE tags IS NULL OR TRIM(tags) = '') AS invalid_tags,
 COUNT(*) FILTER (WHERE source NOT IN ('CNN Indonesia', 'Tempo') OR source IS NULL) AS invalid_source,
 COUNT(*) FILTER (WHERE title_length != length(title) OR content_length != length(content)) AS invalid_lengths,
 COUNT(*) FILTER (WHERE content_length_outlier IS NULL OR iqr_lower IS NULL OR iqr_upper IS NULL OR
                  content_length_outlier != (content_length < iqr_lower OR content_length > iqr_upper)) AS invalid_iqr_flag,
 COUNT(*) - COUNT(DISTINCT (source, title, content_sha256)) AS duplicate_rows
FROM clean_candidate
""").pl()
display(validation)
clean_count = validation.item(0, 'row_count')
assert 0 < clean_count <= raw_count
assert all(validation.item(0, c) == 0 for c in validation.columns if c != 'row_count'), 'Validasi gagal'
stages = list(stage_counts.items()) + [('deduplicated', clean_count)]
audit_rows = []
previous = raw_count
for name, count in stages:
    audit_rows.append({'stage': name, 'rows_remaining': count,
                       'removed_this_stage': previous - count, 'retained_percent': 100 * count / raw_count})
    previous = count
cleaning_audit = pl.DataFrame(audit_rows)
assert cleaning_audit['removed_this_stage'].sum() == raw_count - clean_count
display(cleaning_audit)
clean_profile = con.sql('SUMMARIZE SELECT * FROM clean_candidate').pl()
display(clean_profile)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

row_count,invalid_title,invalid_content,invalid_date,invalid_tags,invalid_source,invalid_lengths,invalid_iqr_flag,duplicate_rows
i64,i64,i64,i64,i64,i64,i64,i64,i64
848028,0,0,0,0,0,0,0,0


stage,rows_remaining,removed_this_stage,retained_percent
str,i64,i64,f64
"""raw""",988814,0,100.0
"""required_text""",974501,14313,98.552508
"""valid_date""",974497,4,98.552104
"""minimum_length""",967113,7384,97.805351
"""deduplicated""",848028,119085,85.762135


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""source""","""VARCHAR""","""CNN Indonesia""","""Tempo""",2,null,null,null,null,null,848028,0.00
"""source_file""","""VARCHAR""","""cnn.csv""","""tempo.csv""",2,null,null,null,null,null,848028,0.00
"""source_row""","""UINTEGER""","""0""","""520863""",544206,"""227313.9712592037""","""139849.04684637245""","""109633""","""219431""","""327482""",848028,0.00
"""published_at""","""TIMESTAMP""","""2014-08-04 08:35:00""","""2020-05-31 16:52:00""",798908,"""2017-11-13 05:34:47.363766""",null,"""2016-06-28 11:09:26.300744""","""2018-01-04 06:05:38.556128""","""2019-04-06 12:06:29.613272""",848028,0.00
"""title""","""VARCHAR""","""""Amuk Munir"" Desak Pemerintah Buka Dokumen TPF Kasus Munir""","""tuhan""",757029,null,null,null,null,null,848028,0.00
"""tags""","""VARCHAR""","""#2019GantiPresiden""","""zurich,asuransi,digital""",630589,null,null,null,null,null,848028,0.00
"""content""","""VARCHAR""",""""" itu,"" kata Nicke Widyawati usai mengecek realisasi penggunaan bahan …","""”Wanhong ini produksinya akan diekspor ke Cina. Karena memang perusaha…",890277,null,null,null,null,null,848028,0.00
"""title_length""","""UINTEGER""","""1""","""126""",97,"""54.302286009424215""","""8.08349505181115""","""50""","""56""","""60""",848028,0.00
"""content_length""","""UINTEGER""","""100""","""52721""",7461,"""2055.199206865811""","""918.9056141574554""","""1518""","""1880""","""2376""",848028,0.00


## 7. Simpan data bersih dan bukti audit

In [8]:
con.close()
CANDIDATE_PATH.replace(OUTPUT_PATH)
audit_tables = {'missing_summary': missing_summary, 'iqr_bounds': iqr_bounds,
                'outlier_audit': outlier_audit, 'overlap_audit': overlap_audit,
                'cleaning_audit': cleaning_audit, 'validation': validation}
for name, table in audit_tables.items():
    table.write_csv(PROCESSED_DIR / f'{name}.csv')
(PROCESSED_DIR / 'cleaning_metadata.json').write_text(json.dumps({
    'pipeline_version': '2026-10-26', 'rows_before': raw_count, 'rows_after': clean_count,
    'dedup_key': DEDUP_KEY, 'min_content_chars': MIN_CONTENT_CHARS,
    'outlier_policy': 'retain_and_flag_per_source_iqr',
}, indent=2), encoding='utf-8')
display(Markdown(f"**Hasil:** {raw_count:,} → {clean_count:,} baris; {raw_count-clean_count:,} baris dikeluarkan. "
                 f"Sebanyak {100*clean_count/raw_count:.2f}% dipertahankan. Semua validasi lulus."))
print('Parquet:', OUTPUT_PATH, f'({OUTPUT_PATH.stat().st_size / 1024**2:.2f} MiB)')


**Hasil:** 988,814 → 848,028 baris; 140,786 baris dikeluarkan. Sebanyak 85.76% dipertahankan. Semua validasi lulus.

Parquet: /home/jovyan/work/data/processed/indonesia_news_clean.parquet (609.97 MiB)


## Catatan Milestone 2

Masalah utama yang diperiksa: kelengkapan isi/tag dan duplikasi antar-file Tempo. Rincian kuantitatif ada pada tabel `missing_summary`, `overlap_audit`, dan `cleaning_audit`. Null pada tags diisi label dengan flag; baris tanpa teks/tanggal wajib dikeluarkan. Teks di bawah 100 karakter dikeluarkan menurut asumsi operasional yang perlu ditinjau mahasiswa. Pencilan IQR dipertahankan karena tidak selalu merupakan kesalahan.

Perubahan dari versi awal: pembacaan CSV ketat, profil awal dan akhir, audit missing beserta persentase, flag IQR, kunci dedup tanpa ketergantungan timestamp, serta validasi yang menghentikan proses bila tidak terpenuhi. Kunci baru dapat mengubah jumlah artikel dan seluruh hasil Milestone 3. Jalankan kembali notebook 03 setelah notebook ini.
